# Import Libraries









In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

from imblearn.over_sampling import SMOTE



# Load Data

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Hotel-A-validation.csv to Hotel-A-validation.csv
Saving Hotel-A-train.csv to Hotel-A-train.csv
Saving Hotel-A-test.csv to Hotel-A-test.csv


In [ ]:
train = pd.read_csv('Hotel-A-train.csv')
validation = pd.read_csv('Hotel-A-validation.csv')
test = pd.read_csv('Hotel-A-test.csv')

# Feature Engineering

In [ ]:
for df in [train, validation, test]:
    df['Booking_date'] = pd.to_datetime(df['Booking_date'])
    df['Expected_checkin'] = pd.to_datetime(df['Expected_checkin'])
    df['Expected_checkout'] = pd.to_datetime(df['Expected_checkout'])

    df['Lead_Time'] = (df['Expected_checkin'] - df['Booking_date']).dt.days
    df['Length_of_Stay'] = (df['Expected_checkout'] - df['Expected_checkin']).dt.days
    df['Revenue'] = df['Room_Rate'] * df['Length_of_Stay']

# Data Cleaning

In [ ]:
train['Reservation_Status'] = train['Reservation_Status'].replace({
    'Check-out': 'Check-Out'
})

validation['Reservation_Status'] = validation['Reservation_Status'].replace({
    'Check-In': 'Check-Out'
})

train      = train[(train['Lead_Time'] >= 0) & (train['Length_of_Stay'] > 0)]
validation = validation[(validation['Lead_Time'] >= 0) & (validation['Length_of_Stay'] > 0)]
test       = test[(test['Lead_Time'] >= 0) & (test['Length_of_Stay'] > 0)]

# Drop Unnecessary Columns

In [ ]:
drop_cols = ['Reservation-id', 'Booking_date', 'Expected_checkin', 'Expected_checkout']

train = train.drop(columns=drop_cols)
validation = validation.drop(columns=drop_cols)
test = test.drop(columns=drop_cols)

# Split Features & Target

In [ ]:
y_train = train['Reservation_Status']
y_val = validation['Reservation_Status']

X_train = train.drop(columns=['Reservation_Status'])
X_val = validation.drop(columns=['Reservation_Status'])

# Encode Categorical Variables

In [ ]:
all_X = pd.concat([X_train, X_val, test], ignore_index=True)
cat_cols = X_train.select_dtypes(include='object').columns

for col in cat_cols:
    le_feat = LabelEncoder()
    le_feat.fit(all_X[col].astype(str))
    X_train[col] = le_feat.transform(X_train[col].astype(str))
    X_val[col] = le_feat.transform(X_val[col].astype(str))
    test[col] = le_feat.transform(test[col].astype(str))

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)

print("Classes:", le.classes_)
print("X_train shape:", X_train.shape)

Classes: ['Canceled' 'Check-Out' 'No-Show']
X_train shape: (26993, 22)


# Handle Class Imbalance (SMOTE)

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Feature Scaling (Logistic Regression)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_res)
X_val_scaled = scaler.transform(X_val)

# Model Evaluation Function

In [ ]:
results = []

def evaluate_model(name, model, X_tr, y_tr, X_v, y_v):
    model.fit(X_tr, y_tr)

    y_pred = model.predict(X_v)
    y_prob = model.predict_proba(X_v)

    acc = accuracy_score(y_v, y_pred)
    prec = precision_score(y_v, y_pred, average='macro')
    rec = recall_score(y_v, y_pred, average='macro')
    f1 = f1_score(y_v, y_pred, average='macro')
    roc = roc_auc_score(y_v, y_prob, multi_class='ovr')

    results.append([name, acc, prec, rec, f1, roc])

# Logistic Regression Model

In [ ]:
lr = LogisticRegression(max_iter=2000, class_weight='balanced')

evaluate_model("Logistic Regression",
               lr,
               X_train_scaled, y_train_res,
               X_val_scaled, y_val)

# Decision Tree Model

In [ ]:
dt = DecisionTreeClassifier(max_depth=10, random_state=42, class_weight='balanced')

evaluate_model("Decision Tree",
               dt,
               X_train_res, y_train_res,
               X_val, y_val)

# Random Forest Model

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    class_weight='balanced'
)

evaluate_model("Random Forest",
               rf,
               X_train_res, y_train_res,
               X_val, y_val)

# Model Comparison Results

In [ ]:
results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"])
results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.465789,0.323919,0.324165,0.317268,0.492314
1,Decision Tree,0.438712,0.326770,0.323366,0.319050,0.493654
2,Random Forest,0.535309,0.342021,0.337675,0.311520,0.506282


# Best Model Selection

In [ ]:
PRIMARY_METRIC = "F1 Score"

best_row = results_df.loc[results_df[PRIMARY_METRIC].idxmax()]
best_model_name = best_row["Model"]

print("Best Model:", best_model_name)
print(best_row)

Best Model: Decision Tree
Model        Decision Tree
Accuracy          0.438712
Precision          0.32677
Recall            0.323366
F1 Score           0.31905
ROC-AUC           0.493654
Name: 1, dtype: object


# Final Model Training

In [ ]:
if best_model_name == "Logistic Regression":
    final_model = LogisticRegression(max_iter=2000, class_weight='balanced')
    final_model.fit(X_train_scaled, y_train_res)
    X_test_final = scaler.transform(test)

elif best_model_name == "Decision Tree":
    final_model = DecisionTreeClassifier(max_depth=10, random_state=42, class_weight='balanced')
    final_model.fit(X_train_res, y_train_res)
    X_test_final = test

else:
    final_model = RandomForestClassifier(n_estimators=300, max_depth=None,
                                         min_samples_split=5, min_samples_leaf=2,
                                         random_state=42, class_weight='balanced')
    final_model.fit(X_train_res, y_train_res)
    X_test_final = test

# Test Prediction

In [ ]:
test_pred = final_model.predict(X_test_final)
test_pred_labels = le.inverse_transform(test_pred)

print(test_pred_labels[:10])

['No-Show' 'No-Show' 'No-Show' 'Check-Out' 'Check-Out' 'Check-Out'
 'Canceled' 'No-Show' 'No-Show' 'Check-Out']


# Confusion Matrix

In [ ]:
if best_model_name == "Logistic Regression":
    y_pred_best = final_model.predict(X_val_scaled)
else:
    y_pred_best = final_model.predict(X_val)

from sklearn.metrics import confusion_matrix
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_best))

Confusion Matrix:
[[ 122  441  175]
 [ 261 1008  333]
 [  72  252   69]]


# Feature Importance

In [ ]:
if best_model_name in ["Decision Tree", "Random Forest"]:
    importances = pd.Series(final_model.feature_importances_, index=X_train.columns)
    print(importances.sort_values(ascending=False).head(10))

Lead_Time            0.147465
Deposit_type         0.135217
Visted_Previously    0.113690
Country_region       0.078965
Ethnicity            0.076565
Hotel_Type           0.064523
Gender               0.051202
Children             0.038489
Revenue              0.038378
Educational_Level    0.035559
dtype: float64
